# PPD — Post-hoc Popularity Debiasing, on Gowalla / Yelp2018

**Paper:** Md Aminul Islam, Elena Zheleva, Ren Wang. *"Post-hoc Popularity Bias
Correction in GNN-based Collaborative Filtering."* WWW '26 (arXiv:2510.12959).

This notebook reproduces PPD on **the exact same Gowalla/Yelp2018 pipeline**
as your own post-hoc BCPS pipeline (`bpsd-gowalla-yelp.ipynb`): same raw-data
staging (10-core filtering, implicit treatment, `ml-1m`-shaped
`ratings.dat`/`movies.dat`), same `DATASET_NAME` switch, same
`build_splits()`/`read_splits()`, same balancing/recycling logic, same
exclusion-set construction, same `LightGCN` backbone, same training loop,
same `evaluate()`, same `TOP_KS = [20, 50, 100]`, same tail definition, and
the same dataset-aware PPAC reference table. Nothing about the data or the
backbone is changed — **CELL 1** and **CELL 2** below are copied *verbatim,
unmodified* from your notebook.

* Set `DATASET_NAME = 'gowalla'` or `'yelp2018'` at the top of **CELL 1**
  (same switch your pipeline uses), then run CELL 1 → CELL 2 → CELL PPD-1 →
  CELL PPD-2. Re-running CELL 1 with the other `DATASET_NAME` and re-running
  everything after it reproduces the other dataset.
* **CELL PPD-1 / CELL PPD-2** implement PPD (Algorithm 1 of the paper) as a
  *post-hoc* correction applied to the SAME trained backbone (`backbone`,
  `sparse_adj`), evaluated with your notebook's own `evaluate()`,
  `EXCL_VAL`/`EXCL_TEST`, `TAIL_VAL`/`TAIL_TEST`, and `item_pop`. These two
  cells are **completely dataset-agnostic** (identical code for ML-1M,
  Gowalla, and Yelp2018) because they only touch the generic variables
  (`train_records`, `NUM_USERS`, `NUM_ITEMS`, ...) that CELL 1/2 already
  produce in the same shape for all three datasets.


## Dataset adaptation notes (read before citing results)

This notebook now runs on **either Gowalla or Yelp2018** via the `DATASET_NAME`
switch at the top of Cell 1 (`'gowalla'` or `'yelp2018'`). `loc-gowalla_edges.txt`
(Gowalla's friendship graph) and Yelp's `user.json`/`tip.json`/`checkin.json` are
not used anywhere in this pipeline.

Stage 1 onward (LightGCN backbone, BCPS mechanisms, the regression/centroid
basis, the behaviour-conditioned gates, the sweep/controls/probe) is
**unmodified for both datasets** — those cells operate on `item_map`,
`train_records`, `interactions_df`, `content`/`category` generically and do
not know which dataset produced them. Three things change per dataset, each
flagged with a `DATASET ADAPTATION` comment at the point it occurs:

1. **Cell 1 — data loading.** Both datasets ship raw and un-split, so a
   one-off preprocessing pass converts whichever is selected into ml-1m's
   `ratings.dat`/`movies.dat` on-disk shape before `build_splits()` /
   `read_splits()` run — those two functions are byte-identical across all
   three datasets (ml-1m, Gowalla, Yelp2018). The standard 10-core filter
   used throughout the graph-CF literature for these two benchmarks is
   applied once, iteratively, before anything else sees the data.
2. **Cell 1 — `POSITIVE_RATING_THRESHOLD`.** Both Gowalla and Yelp2018 are
   treated as fully **implicit**: every check-in / every review is kept as
   a positive regardless of any star rating. This matches how PPAC's own
   released code (github.com/Stevenn9981/PPAC) trains on these two datasets
   — both come from LightGCN's standard pre-built implicit splits, where
   PPAC's `rating >= 4` thresholding is applied *only* to their ML-1M run.
   Applying the ML-1M-style threshold to Yelp would silently produce a
   different (and non-comparable) dataset from PPAC's own Yelp2018 numbers.
3. **Cell 3 — `build_item_metadata`.**
   - *Yelp2018*: businesses genuinely have multi-label categories (Yelp's
     own `categories` field), so this reuses the **original genre-based
     logic almost verbatim** — categories in place of genres, same
     multi-hot + KMeans construction. No conceptual substitution needed.
   - *Gowalla*: locations have no genre-equivalent field, so content
     vectors/categories come from each location's mean check-in
     coordinates instead — KMeans over standardized (lat, lon) stands in
     for KMeans over genre multi-hot vectors, producing geographic regions.
     The §7.2 diversity proxy (`1 - cosine`) is still well-defined on the
     resulting content vectors, but should be read as "directional
     similarity in standardized geographic space," not genre overlap.

**Cell 6 — PPAC comparison is now dataset-aware.** `PPAC_REFERENCE` holds
PPAC's published LightGCN base/PPAC numbers for all three of their datasets
(transcribed from arXiv:2402.07425, Table 2), and the notebook picks the row
matching `DATASET_NAME` automatically, so the "vs PPAC" section is a valid
comparison whichever dataset this notebook is run with — it no longer
silently compares Gowalla/Yelp2018 results against ML-1M's numbers.

**One methodological note worth stating explicitly in the write-up:** on
Yelp2018 the true 1–5 star rating is preserved in the staged `ratings.dat`
(unlike Gowalla, which has no rating at all and uses a 1.0 placeholder), so
`LOYALTY_WEIGHT = 'rating'` is a meaningful option there if wanted — the
default (`'count'`) ignores it and weights every interaction equally,
consistent with the fully-implicit treatment described in point 2 above.
Also: Cell 1's Yelp loader drops businesses with no `categories` field
before building the interaction graph (Stage 2 needs a category for every
surviving item); only a small fraction of Yelp businesses lack this field,
but it is a small deviation from the canonical "keep every review" Yelp2018
preprocessing and is worth a one-line mention in a methods section.

In [1]:
# ============================================================================
# CELL 1 — CONFIG, SPLITS, EXCLUSION SETS
#
# DATASET ADAPTATION (was: MovieLens-1M).
# Everything below build_splits()/read_splits() is written against ml-1m's
# on-disk shape: a 'ratings.dat' (user::item::rating::timestamp) and a
# 'movies.dat' (item::<meta>::<meta>), both '::'-delimited. Gowalla and
# Yelp2018 ship in very different raw shapes, so a one-off preprocessing
# pass below converts whichever is selected (DATASET_NAME) into that same
# shape and points RAW_DATA_DIR at the result -- build_splits(),
# read_splits() and exclusion_for() then run BYTE-IDENTICAL to the ML-1M
# version, for either dataset.
#
#   * Gowalla (SNAP loc-gowalla_totalCheckins.txt[.gz]): raw, un-split,
#     timestamped check-ins, no rating, no item metadata. Every check-in is
#     an implicit positive; content/category come from each location's
#     mean check-in coordinates (see Cell 3 -- no genre-equivalent exists).
#   * Yelp2018 (Yelp Open Dataset business.json + review.json): every
#     review is an implicit positive REGARDLESS of its star rating -- this
#     matches how PPAC's own released code trains on Gowalla/Yelp2018 (both
#     come from LightGCN's standard pre-built implicit splits, where every
#     interaction counts; only their ML-1M run thresholds by rating >= 4).
#     Businesses DO have genuine multi-label categories, so Cell 3 reuses
#     the ORIGINAL genre-based build_item_metadata almost verbatim for this
#     dataset (categories in place of genres) -- no lat/lon workaround
#     needed here.
# In both cases the true rating (Gowalla: none, written as a 1.0
# placeholder; Yelp: the real 1-5 star count) is still carried through
# ratings.dat so LOYALTY_WEIGHT='rating' remains available if wanted.
# ============================================================================
import os, sys, time, math, json, copy, random, collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

DATASET_NAME = 'gowalla'   # 'gowalla' | 'yelp2018'

# raw Gowalla/Yelp are both far too sparse for collaborative filtering
# without this -- the standard 10-core filter used for both benchmarks
# throughout the graph-CF literature (e.g. NGCF/LightGCN's splits: Gowalla
# ~29.9k users / ~41.0k items, Yelp2018 ~31.7k users / ~38.0k items).
# Applied ONCE, iteratively, before anything else in the notebook sees the
# data.
MIN_USER_INTERACTIONS = 10
MIN_ITEM_INTERACTIONS = 10


def _kcore_filter(edges, user_col='user', item_col='item'):
    """Iterative bipartite k-core filter, shared by both dataset loaders."""
    while True:
        uc = edges[user_col].value_counts()
        ic = edges[item_col].value_counts()
        keep_u = uc.index[uc >= MIN_USER_INTERACTIONS]
        keep_i = ic.index[ic >= MIN_ITEM_INTERACTIONS]
        filtered = edges[edges[user_col].isin(keep_u) & edges[item_col].isin(keep_i)]
        if len(filtered) == len(edges):
            return filtered
        edges = filtered


# ---------------------------------------------------------------------------
# GOWALLA loader
# ---------------------------------------------------------------------------
GOWALLA_DIR = '/kaggle/input/datasets/marquis03/gowalla'   # folder containing loc-gowalla_totalCheckins.txt(.gz)


def _find_gowalla_checkins():
    candidates = [
        GOWALLA_DIR, os.environ.get('GOWALLA_DIR'),
        '/kaggle/input/gowalla', '/kaggle/input/gowalla-dataset',
        '/kaggle/input/loc-gowalla', '/kaggle/input/snap-gowalla',
        './gowalla', './data/gowalla', '../gowalla',
    ]
    names = ['loc-gowalla_totalCheckins.txt.gz', 'loc-gowalla_totalCheckins.txt',
              'Gowalla_totalCheckins.txt.gz', 'Gowalla_totalCheckins.txt']
    for c in candidates:
        if not c:
            continue
        for n in names:
            p = os.path.join(c, n)
            if os.path.isfile(p):
                return os.path.abspath(p)
    found = []
    if os.path.isdir('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            for f in files:
                fl = f.lower()
                if 'gowalla' in fl and 'checkin' in fl:
                    found.append(os.path.join(root, f))
    if found:
        return os.path.abspath(sorted(found)[0])
    raise FileNotFoundError(
        'Could not find loc-gowalla_totalCheckins.txt(.gz). Add the dataset via '
        '"+ Add Input" first, or set GOWALLA_DIR.')


def _prepare_gowalla_raw(checkins_path, out_root):
    """Raw check-ins -> ml-1m-shaped ratings.dat + movies.dat.

    ratings.dat : user::item::1.0::timestamp   (implicit; rating is a placeholder)
    movies.dat  : item::mean_lat::mean_lon      (in place of item::title::genres;
                  Cell 3's build_item_metadata reads these two floats instead of
                  a genre string when DATASET_NAME == 'gowalla').

    One interaction per (user, item): repeat check-ins to the same location are
    collapsed to a single positive, timestamped at the EARLIEST check-in to that
    location (first exposure).
    """
    print(f'[gowalla] reading {checkins_path}')
    raw = pd.read_csv(checkins_path, sep='\t', header=None,
                       names=['user', 'time', 'lat', 'lon', 'item'],
                       dtype={'user': np.int64, 'lat': np.float64,
                              'lon': np.float64, 'item': np.int64})
    raw['ts'] = (pd.to_datetime(raw['time'], format='%Y-%m-%dT%H:%M:%SZ', utc=True)
                 .astype('int64') // 10**9)
    print(f'[gowalla] raw check-ins: {len(raw):,} rows, '
          f'{raw["user"].nunique():,} users, {raw["item"].nunique():,} locations')

    coords = raw.groupby('item')[['lat', 'lon']].mean()
    edges = raw.groupby(['user', 'item'], as_index=False)['ts'].min()
    edges = _kcore_filter(edges)
    print(f'[gowalla] after {MIN_USER_INTERACTIONS}-core filtering: '
          f'{len(edges):,} interactions, {edges["user"].nunique():,} users, '
          f'{edges["item"].nunique():,} items')

    staging_dir = os.path.join(out_root, 'gowalla_raw')
    os.makedirs(staging_dir, exist_ok=True)

    items_sorted = np.sort(edges['item'].unique())
    with open(os.path.join(staging_dir, 'movies.dat'), 'w', encoding='latin-1') as fh:
        for it in items_sorted:
            lat, lon = coords.loc[it]
            fh.write(f'{it}::{lat:.6f}::{lon:.6f}\n')

    with open(os.path.join(staging_dir, 'ratings.dat'), 'w', encoding='latin-1') as fh:
        for u, i, t in zip(edges['user'].to_numpy(), edges['item'].to_numpy(),
                            edges['ts'].to_numpy()):
            fh.write(f'{u}::{i}::1.0::{t}\n')

    print(f'[gowalla] staged ml-1m-shaped files in {staging_dir}')
    return staging_dir


# ---------------------------------------------------------------------------
# YELP2018 loader
# ---------------------------------------------------------------------------
YELP_DIR = '/kaggle/input/datasets/yelp-dataset/yelp-dataset'   # folder containing the Yelp Open Dataset json files


def _find_yelp_files():
    candidates = [
        YELP_DIR, os.environ.get('YELP_DIR'),
        '/kaggle/input/yelp-dataset', '/kaggle/input/yelp',
        './yelp', './data/yelp', '../yelp',
    ]
    biz_names = ['yelp_academic_dataset_business.json', 'business.json']
    rev_names = ['yelp_academic_dataset_review.json', 'review.json']

    def _find(names):
        for c in candidates:
            if not c:
                continue
            for n in names:
                p = os.path.join(c, n)
                if os.path.isfile(p):
                    return os.path.abspath(p)
        if os.path.isdir('/kaggle/input'):
            for root, _, files in os.walk('/kaggle/input'):
                for f in files:
                    if f in names:
                        return os.path.abspath(os.path.join(root, f))
        return None

    biz, rev = _find(biz_names), _find(rev_names)
    missing = [n for n, p in [('business.json', biz), ('review.json', rev)] if p is None]
    if missing:
        raise FileNotFoundError(
            f'Could not find Yelp Open Dataset file(s): {missing}. Add the dataset '
            f'via "+ Add Input" first, or set YELP_DIR.')
    return biz, rev


def _prepare_yelp_raw(business_path, review_path, out_root):
    """Raw Yelp Open Dataset JSON -> ml-1m-shaped ratings.dat + movies.dat.

    ratings.dat : user::item::stars::timestamp  (implicit: EVERY review kept,
                  regardless of star count -- see the DATASET ADAPTATION note
                  above for why; the true 1-5 star value is still written, in
                  case LOYALTY_WEIGHT='rating' is wanted later)
    movies.dat  : item::item::cat1|cat2|...      (business_id repeated as the
                  unused "title" field; categories pipe-joined exactly like
                  ml-1m's genre string -- Cell 3's ORIGINAL, unmodified
                  build_item_metadata reads this directly)

    user_id/business_id in the raw JSON are opaque strings. Item ids can stay
    as raw strings (item_map is keyed by string throughout, same as ml-1m's
    numeric-string movie ids). User ids get remapped to sequential integers
    here, because build_splits() does `int(u)` on the first ratings.dat field
    (harmless: read_splits() remaps everything to compact 0-based ids again
    right after, same as it does for ml-1m/Gowalla's native integer ids).

    Businesses with no `categories` are dropped up front (Stage 2 needs a
    category for every surviving item; only a small fraction of Yelp
    businesses lack this field).
    """
    print(f'[yelp] reading {business_path}')
    cat_of = {}
    with open(business_path, encoding='utf-8') as fh:
        for line in fh:
            d = json.loads(line)
            c = d.get('categories')
            if not c:
                continue
            toks = [t.strip().replace(':', ' ') for t in c.split(',') if t.strip()]
            if toks:
                cat_of[d['business_id']] = '|'.join(toks)
    print(f'[yelp] businesses with categories: {len(cat_of):,}')

    print(f'[yelp] reading {review_path} (this can take a few minutes)')
    users, items, stars, dates = [], [], [], []
    with open(review_path, encoding='utf-8') as fh:
        for line in fh:
            d = json.loads(line)
            bid = d['business_id']
            if bid not in cat_of:          # need a category for Stage 2
                continue
            users.append(d['user_id']); items.append(bid)
            stars.append(int(d['stars'])); dates.append(d['date'])
    raw = pd.DataFrame({'user': users, 'item': items, 'stars': stars, 'date': dates})
    del users, items, stars, dates
    print(f'[yelp] raw reviews (categorised businesses only): {len(raw):,} rows, '
          f'{raw["user"].nunique():,} users, {raw["item"].nunique():,} businesses')

    raw['ts'] = (pd.to_datetime(raw['date'], format='%Y-%m-%d %H:%M:%S', utc=True)
                 .astype('int64') // 10**9)

    # one row per (user, item): keep the EARLIEST review if a pair repeats (rare).
    edges = (raw.sort_values('ts')
             .groupby(['user', 'item'], as_index=False)
             .first()[['user', 'item', 'stars', 'ts']])
    edges = _kcore_filter(edges)
    print(f'[yelp] after {MIN_USER_INTERACTIONS}-core filtering: '
          f'{len(edges):,} interactions, {edges["user"].nunique():,} users, '
          f'{edges["item"].nunique():,} items')

    staging_dir = os.path.join(out_root, 'yelp_raw')
    os.makedirs(staging_dir, exist_ok=True)

    user_id_map = {u: idx for idx, u in enumerate(sorted(edges['user'].unique()))}
    items_sorted = sorted(edges['item'].unique())
    with open(os.path.join(staging_dir, 'movies.dat'), 'w', encoding='utf-8') as fh:
        for b in items_sorted:
            fh.write(f'{b}::{b}::{cat_of[b]}\n')

    with open(os.path.join(staging_dir, 'ratings.dat'), 'w', encoding='utf-8') as fh:
        for u, i, s, t in zip(edges['user'].to_numpy(), edges['item'].to_numpy(),
                               edges['stars'].to_numpy(), edges['ts'].to_numpy()):
            fh.write(f'{user_id_map[u]}::{i}::{float(s):.1f}::{t}\n')

    print(f'[yelp] staged ml-1m-shaped files in {staging_dir}')
    return staging_dir


# /kaggle/input is READ-ONLY, so every output (including the staged files)
# goes to /kaggle/working.
OUT_ROOT = ('/kaggle/working/bpsd_out' if os.path.isdir('/kaggle/working')
            else os.environ.get('BPSD_OUT', './bpsd_out'))
os.makedirs(OUT_ROOT, exist_ok=True)

if DATASET_NAME == 'gowalla':
    RAW_DATA_DIR = _prepare_gowalla_raw(_find_gowalla_checkins(), OUT_ROOT)
elif DATASET_NAME == 'yelp2018':
    _biz, _rev = _find_yelp_files()
    RAW_DATA_DIR = _prepare_yelp_raw(_biz, _rev, OUT_ROOT)
else:
    raise ValueError(f"DATASET_NAME must be 'gowalla' or 'yelp2018', got {DATASET_NAME!r}")

WORKING_DIR = os.path.join(OUT_ROOT, 'dataset', DATASET_NAME)
CKPT_DIR    = os.path.join(OUT_ROOT, 'checkpoints')
os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'[paths] staged raw = {RAW_DATA_DIR}')
print(f'[paths] out        = {os.path.abspath(OUT_ROOT)}')

SEED = 2020
# DATASET ADAPTATION: both Gowalla and Yelp2018 are treated as fully implicit
# (every staged interaction is a positive, regardless of any star/rating
# value carried alongside it) -- matching how PPAC's own released code
# trains on these two datasets (LightGCN's standard pre-built implicit
# splits). Only ML-1M's own notebook thresholds by rating >= 4.
POSITIVE_RATING_THRESHOLD = 1.0
TEST_HOLDOUT_PER_USER     = 30    # PPAC's TOP_K -- protocol kept identical
BALANCE_PER_ITEM          = None  # None -> derive from this split's own holdout pool
VAL_HOLDOUT_PER_USER      = 20    # only used when STRICT_PPAC = False

# STRICT_PPAC = True  -> the paper's released protocol: no validation split,
#                        checkpoint selected by NDCG@50 on the test set.
#                        Use ONLY for the head-to-head against Table 2.
# STRICT_PPAC = False -> carve a validation split and select on it. Honest.
STRICT_PPAC = False

# PPAC Sec 4.1: "...another 10% as the validation set using the same way.
# The remaining interactions are used for training."  i.e. PPAC has NO orphaned
# interactions -- anything not sampled into the balanced test/val is TRAINING
# data. Balancing here discards held-out positives; leaving them in limbo
# (neither trained on nor masked) is what inflated the old numbers.
RECYCLE_DISCARDS = True   # put balancing's discards back into train, per PPAC

EVAL_ON_BALANCED = True   # True  -> balanced (intervened) test set == paper's protocol
                          # False -> the raw 30-per-user holdout    == biased test set

TOP_KS        = [20, 50, 100]
TAIL_FRACTION = 1.0 / 3.0   # bottom third of the EVALUATED item universe, by train popularity
SAVE_CHECKPOINTS = False    # opt-in; nothing in this notebook ever READS a .pt
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[device] {DEVICE}')


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)


def build_splits():
    """Reproduce PPAC's split protocol, plus its balanced (intervened) test set.

    PPAC (run_MF.py::create_train_and_test):
      * positives are ratings > 3  (here: every staged interaction -- see
        POSITIVE_RATING_THRESHOLD above)
      * every user with MORE than 30 positives contributes exactly 30 to test
      * every other user contributes all of their positives to train
      * NO global minimum-interaction user filter, NO validation split

    Balanced (intervened) set: fix a per-item quota n, keep every item with at
    least n held-out interactions, subsample exactly n. Every retained item then
    contributes the same number of test interactions.

    n is not a free constant. Retained = n * |{i : c_i >= n}| has an interior
    maximum: small n throws away interactions from popular items, large n throws
    away items entirely. BALANCE_PER_ITEM = None picks the argmax from THIS
    split's own item counts.

    Built entirely from ratings.dat and movies.dat (here: the staged files for
    whichever DATASET_NAME was selected). No external split files.
    """
    rng = random.Random(SEED)
    by_user = collections.defaultdict(list)
    rating_of, ts_of = {}, {}

    with open(os.path.join(RAW_DATA_DIR, 'ratings.dat'), encoding='latin-1') as fh:
        for line in fh:
            u, i, r, t = line.strip().split('::')
            u, r, t = int(u), float(r), int(t)
            if r >= POSITIVE_RATING_THRESHOLD:
                by_user[u].append(i)
                rating_of[(u, i)] = r
                ts_of[(u, i)] = t

    train, val, test_pool = (collections.defaultdict(list) for _ in range(3))
    for u, items in by_user.items():
        items = list(items)
        rng.shuffle(items)
        need = TEST_HOLDOUT_PER_USER + (0 if STRICT_PPAC else VAL_HOLDOUT_PER_USER)
        if len(items) > need:
            test_pool[u] = items[:TEST_HOLDOUT_PER_USER]
            if not STRICT_PPAC:
                val[u] = items[TEST_HOLDOUT_PER_USER:need]
            train[u] = items[need:]
        else:
            train[u] = items

    def _balance(pool, quota=None):
        hits = collections.defaultdict(list)
        for u, items in pool.items():
            for i in items:
                hits[i].append(u)
        counts = np.array(sorted(len(v) for v in hits.values()))
        retained = counts[::-1] * (np.arange(len(counts)) + 1)
        q = int(quota) if quota else int(counts[::-1][retained.argmax()])
        out, n_items = collections.defaultdict(list), 0
        for i, users in hits.items():
            if len(users) >= q:
                n_items += 1
                for u in rng.sample(users, q):
                    out[u].append(i)
        return out, q, n_items

    balanced, quota, n_bal_items = _balance(test_pool, BALANCE_PER_ITEM)
    if STRICT_PPAC:
        bal_val, vq, vn = collections.defaultdict(list), 0, 0
    else:
        bal_val, vq, vn = _balance(val)

    print(f'[split] users={len(by_user)}')
    print(f'[split] raw test: {len(test_pool)} users, '
          f'{sum(map(len, test_pool.values()))} interactions')
    print(f'[split] balanced test: {len(balanced)} users, '
          f'{sum(map(len, balanced.values()))} interactions over '
          f'{n_bal_items} items @ {quota} each')
    if not STRICT_PPAC:
        print(f'[split] raw val: {len(val)} users, {sum(map(len, val.values()))} interactions')
        print(f'[split] balanced val: {len(bal_val)} users, '
              f'{sum(map(len, bal_val.values()))} interactions over {vn} items @ {vq} each')

    # --- PPAC: "The remaining interactions are used for training." --------------
    # Balancing throws away held-out positives. PPAC has no such category:
    # anything not sampled into the balanced test/val is training data. Leaving
    # them orphaned -- neither trained on nor masked at eval -- is what made a
    # vanilla LightGCN beat PPAC's own published method on ML-1M.
    n_before = sum(map(len, train.values()))
    if RECYCLE_DISCARDS:
        n_rec = 0
        for pool, keepset in ((test_pool, balanced), (val, bal_val)):
            for u, items in pool.items():
                k = set(keepset.get(u, []))
                extra = [i for i in items if i not in k]
                train[u].extend(extra)
                n_rec += len(extra)
        print(f'[split] recycled {n_rec:,} discarded holdout interactions into train '
              f'(PPAC Sec 4.1); train {n_before:,} -> {sum(map(len, train.values())):,}')
    else:
        orphan = (sum(map(len, test_pool.values())) - sum(map(len, balanced.values()))
                  + sum(map(len, val.values())) - sum(map(len, bal_val.values())))
        print(f'[split] WARNING: {orphan:,} held-out positives are in NO split '
              f'(not trained on, not scored). This is not PPAC\'s protocol.')
    print(f'[split] final train_inter={sum(map(len, train.values()))}')

    def dump(name, recs):
        with open(os.path.join(WORKING_DIR, name), 'w', encoding='utf-8') as fh:
            for u, items in recs.items():
                for i in items:
                    fh.write(f'{u}::{i}::{rating_of[(u, i)]:.1f}::{ts_of[(u, i)]}\n')

    dump('ratings.train', train)
    dump('ratings.test', test_pool)
    dump('balance_ratings.test', balanced)
    if not STRICT_PPAC:
        dump('ratings.val', val)
        dump('balance_ratings.val', bal_val)


def read_splits():
    """Load the splits and remap to compact 0-based ids.

    Returns SCORING targets (balanced, restricted to items seen in train) and,
    separately, the RAW holdout pools used only for masking. Keeping these
    apart is the fix for the exclusion bug: balancing discards held-out
    positives, and those discarded items must still be masked out of the
    candidate list even though they are never scored.
    """
    item_map = {}
    with open(os.path.join(RAW_DATA_DIR, 'movies.dat'), encoding='latin-1') as fh:
        for idx, line in enumerate(fh):
            item_map[line.split('::')[0]] = idx

    def load(name):
        recs = collections.defaultdict(list)
        path = os.path.join(WORKING_DIR, name)
        if not os.path.exists(path):
            return recs
        with open(path, encoding='utf-8') as fh:
            for line in fh:
                u, i, _, _ = line.strip().split('::')
                recs[int(u)].append(item_map[i])
        return recs

    raw_train = load('ratings.train')
    user_map = {raw: new for new, raw in enumerate(sorted(raw_train))}
    train = collections.defaultdict(list, {user_map[u]: v for u, v in raw_train.items()})
    train_items = {i for v in train.values() for i in v}

    def remap(name, rankable_only):
        """rankable_only=True  -> scoring target: drop items never seen in train.
           rankable_only=False -> masking pool: keep every known positive."""
        out = collections.defaultdict(list)
        for u, items in load(name).items():
            if u not in user_map:
                continue
            keep = items if not rankable_only else [i for i in items if i in train_items]
            if keep:
                out[user_map[u]] = keep
        return out

    test = remap('balance_ratings.test' if EVAL_ON_BALANCED else 'ratings.test', True)
    val  = remap('balance_ratings.val'  if EVAL_ON_BALANCED else 'ratings.val',  True)

    # FIX: masking pools come from the RAW holdouts, never the balanced subsets.
    raw_val  = remap('ratings.val',  False)   # empty dict when STRICT_PPAC
    raw_test = remap('ratings.test', False)

    if STRICT_PPAC:
        val = test          # paper protocol: selection happens on the eval set
    params = {'num_users': len(user_map), 'num_items': len(item_map)}
    return train, val, test, raw_val, raw_test, user_map, item_map, params


set_seed(SEED)
build_splits()
(train_records, val_records, test_records,
 raw_val_records, raw_test_records, user_map, item_map, params) = read_splits()
NUM_USERS, NUM_ITEMS = params['num_users'], params['num_items']


# --- exclusion = every known positive MINUS the current scoring target -------
# With RECYCLE_DISCARDS=True this is now just the STANDARD rule (mask the user's
# training items, plus val when scoring test) -- because the discards ARE
# training items. The ambiguity that existed before disappears once the split
# stops orphaning them.
def exclusion_for(target):
    """Mask everything the user is known to like, except what we score now."""
    out = {}
    for u in train_records:
        seen = (set(train_records.get(u, []))
                | set(raw_val_records.get(u, []))
                | set(raw_test_records.get(u, [])))
        out[u] = list(seen - set(target.get(u, [])))
    return out


EXCL_VAL  = exclusion_for(val_records)
EXCL_TEST = exclusion_for(test_records)

_per_user_old = np.mean([len(train_records.get(u, [])) + len(val_records.get(u, []))
                         for u in test_records])
_per_user_new = np.mean([len(EXCL_TEST.get(u, [])) for u in test_records])
print(f'[data] users={NUM_USERS} items={NUM_ITEMS} '
      f'train={sum(map(len, train_records.values()))} '
      f'val_users={len(val_records)} test_users={len(test_records)}')
print(f'[mask] mean masked items per test user: {_per_user_new:.1f}')
if RECYCLE_DISCARDS:
    print('[mask] == standard rule (train + val), since discards are now training data')


[gowalla] reading /kaggle/input/datasets/marquis03/gowalla/Gowalla_totalCheckins.txt
[gowalla] raw check-ins: 6,442,892 rows, 107,092 users, 1,280,969 locations
[gowalla] after 10-core filtering: 1,027,464 interactions, 29,858 users, 40,988 items
[gowalla] staged ml-1m-shaped files in /kaggle/working/bpsd_out/gowalla_raw
[paths] staged raw = /kaggle/working/bpsd_out/gowalla_raw
[paths] out        = /kaggle/working/bpsd_out
[device] cuda
[split] users=29858
[split] raw test: 4690 users, 140700 interactions
[split] balanced test: 4690 users, 52785 interactions over 17595 items @ 3 each
[split] raw val: 4690 users, 93800 interactions
[split] balanced val: 4689 users, 39014 interactions over 19507 items @ 2 each
[split] recycled 142,701 discarded holdout interactions into train (PPAC Sec 4.1); train 792,964 -> 935,665
[split] final train_inter=935665
[data] users=29858 items=40988 train=935665 val_users=4689 test_users=4690
[mask] mean masked items per test user: 94.8
[mask] == standard ru

In [2]:
# ============================================================================
# CELL 2 — STAGE 1: LIGHTGCN BACKBONE  (trained from scratch, no checkpoint)
# ============================================================================
LATENT_DIM = 64
N_LAYERS   = 3
LR         = 1e-3       # paper reports 0.01; 1e-3 is the released-code default
REG_WEIGHT = 1e-4
BATCH_SIZE = 8192
EPOCHS     = 2000
PATIENCE   = 50


def build_sparse_adj(train_recs, num_users, num_items):
    n = num_users + num_items
    us, it = [], []
    for u, items in train_recs.items():
        us.extend([u] * len(items))
        it.extend([i + num_users for i in items])
    us = torch.tensor(us, dtype=torch.long)
    it = torch.tensor(it, dtype=torch.long)
    src = torch.cat([us, it]); dst = torch.cat([it, us])
    deg = torch.zeros(n).scatter_add_(0, dst, torch.ones(dst.numel()))
    dis = deg.clamp(min=1.0).pow(-0.5)
    return torch.sparse_coo_tensor(
        torch.stack([dst, src]), dis[dst] * dis[src], (n, n)).coalesce()


def propagate(adj, e_u0, e_i0, n_layers=N_LAYERS):
    """LightGCN readout: mean over layers 0..L (alpha_l = 1/(L+1))."""
    x = torch.cat([e_u0, e_i0], dim=0)
    layers = [x]
    for _ in range(n_layers):
        x = torch.sparse.mm(adj, x)
        layers.append(x)
    out = torch.stack(layers, dim=1).mean(dim=1)
    return torch.split(out, [e_u0.shape[0], e_i0.shape[0]], dim=0)


def sample_triplets(train_recs, num_items):
    users, pos = [], []
    for u, items in train_recs.items():
        users.extend([u] * len(items)); pos.extend(items)
    users = np.asarray(users, dtype=np.int64); pos = np.asarray(pos, dtype=np.int64)
    psets = {u: set(v) for u, v in train_recs.items()}
    neg = np.random.randint(0, num_items, size=len(users), dtype=np.int64)
    bad = np.fromiter((n in psets[u] for u, n in zip(users, neg)), bool, len(users))
    while bad.any():
        idx = np.flatnonzero(bad)
        neg[idx] = np.random.randint(0, num_items, size=len(idx), dtype=np.int64)
        bad[idx] = np.fromiter((neg[j] in psets[users[j]] for j in idx), bool, len(idx))
    return torch.from_numpy(users), torch.from_numpy(pos), torch.from_numpy(neg)


def recall_ndcg(ground_truth, ranked, k):
    disc = 1.0 / np.log2(np.arange(2, k + 2))
    rec, ndcg = [], []
    for truth, row_ in zip(ground_truth, ranked):
        ts = set(truth)
        hits = np.fromiter((i in ts for i in row_[:k]), np.float32, k)
        rec.append(hits.sum() / len(ts))
        idcg = disc[:min(len(ts), k)].sum()
        ndcg.append(float((hits * disc).sum()) / idcg if idcg > 0 else 0.0)
    return float(np.mean(rec)), float(np.mean(ndcg))


# --- FIX #3: tail defined INSIDE the evaluated item universe ------------------
# The old definition (item_pop <= median over all 3883 items) is empty by
# construction under EVAL_ON_BALANCED: the balanced set retains only items with
# >= 67 held-out interactions, every one of which is far above median. The
# metric returned a hardcoded 0.0 no matter what the model did.
def tail_items_for(eval_recs, pop, frac=TAIL_FRACTION):
    universe = np.array(sorted({i for v in eval_recs.values() for i in v}), dtype=np.int64)
    if universe.size == 0:
        return set(), 0.0
    pops = pop[universe]
    cutoff = float(np.quantile(pops, frac))
    return set(universe[pops <= cutoff].tolist()), cutoff


@torch.no_grad()
def evaluate(e_u, e_i, eval_recs, exclude_recs, pop, ks=TOP_KS, tail_set=None):
    users = sorted(u for u, v in eval_recs.items() if v)
    max_k = max(ks)
    ranked_all = []
    for s in range(0, len(users), 512):
        batch = users[s:s + 512]
        scores = e_u[batch] @ e_i.T
        for r, u in enumerate(batch):
            ex = exclude_recs.get(u, [])
            if ex:
                scores[r, torch.tensor(ex, dtype=torch.long, device=scores.device)] = -torch.inf
        ranked_all.append(torch.topk(scores, k=max_k, dim=1).indices.cpu().numpy())
    ranked = np.concatenate(ranked_all, 0)
    truth = [eval_recs[u] for u in users]

    out = {}
    for k in ks:
        r, n = recall_ndcg(truth, ranked, k)
        out[k] = {'recall': r, 'ndcg': n, 'arp': float(pop[ranked[:, :k]].mean())}

    if tail_set is None:
        tail_set, _ = tail_items_for(eval_recs, pop)
    tt = [[i for i in t if i in tail_set] for t in truth]
    keep = [j for j, t in enumerate(tt) if t]
    out['tail_recall@50'] = (recall_ndcg([tt[j] for j in keep], ranked[keep], 50)[0]
                             if keep else float('nan'))
    out['tail_users'] = len(keep)
    return out


class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, dim=LATENT_DIM):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, dim)
        self.item_embedding = nn.Embedding(num_items, dim)
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def readout(self, adj):
        return propagate(adj, self.user_embedding.weight, self.item_embedding.weight)


def item_popularity_from(train_recs, num_items):
    c = np.zeros(num_items, dtype=np.float32)
    for items in train_recs.values():
        c[np.asarray(items, dtype=np.int64)] += 1.0
    return c / max(float(c.max()), 1.0)


def train_backbone():
    """Trains LightGCN from random init. Best state is kept IN MEMORY."""
    set_seed(SEED)
    adj = build_sparse_adj(train_records, NUM_USERS, NUM_ITEMS).to(DEVICE)
    model = LightGCN(NUM_USERS, NUM_ITEMS).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    pop = item_popularity_from(train_records, NUM_ITEMS)
    tail_val, cut_v = tail_items_for(val_records, pop)
    tail_test, cut_t = tail_items_for(test_records, pop)
    print(f'[tail] val: {len(tail_val)} items below pop {cut_v:.4f} | '
          f'test: {len(tail_test)} items below pop {cut_t:.4f}')

    best, best_ep, best_state = -np.inf, -1, None
    for epoch in range(EPOCHS):
        model.train(); t0 = time.time()
        u, p, n = sample_triplets(train_records, NUM_ITEMS)
        perm = torch.randperm(len(u)); u, p, n = u[perm], p[perm], n[perm]
        tot, nb = 0.0, 0
        for s in range(0, len(u), BATCH_SIZE):
            ub  = u[s:s+BATCH_SIZE].to(DEVICE)
            pb  = p[s:s+BATCH_SIZE].to(DEVICE)
            nb_ = n[s:s+BATCH_SIZE].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            eu, ei = model.readout(adj)
            pos = (eu[ub] * ei[pb]).sum(1)
            neg = (eu[ub] * ei[nb_]).sum(1)
            rank = F.softplus(neg - pos).mean()
            ego = (model.user_embedding(ub).pow(2).sum()
                   + model.item_embedding(pb).pow(2).sum()
                   + model.item_embedding(nb_).pow(2).sum()) / (2.0 * len(ub))
            loss = rank + REG_WEIGHT * ego
            loss.backward(); opt.step()
            tot += loss.item(); nb += 1

        model.eval()
        with torch.no_grad():
            eu, ei = model.readout(adj)
            m = evaluate(eu, ei, val_records, EXCL_VAL, pop, tail_set=tail_val)
        if m[50]['ndcg'] > best:
            best, best_ep = m[50]['ndcg'], epoch
            best_state = copy.deepcopy(model.state_dict())
        print(f'Epoch [{epoch+1}/{EPOCHS}] Loss {tot/nb:.4f} '
              f'Recall@50 {m[50]["recall"]:.4f} NDCG@50 {m[50]["ndcg"]:.4f} '
              f'ARP@50 {m[50]["arp"]:.4f} {time.time()-t0:.1f}s')
        if epoch - best_ep >= PATIENCE:
            print(f'Early stop at {epoch+1}; best epoch {best_ep+1}'); break

    model.load_state_dict(best_state)
    model.eval()
    if SAVE_CHECKPOINTS:
        torch.save({'epoch': best_ep, 'model_state_dict': best_state},
                   os.path.join(CKPT_DIR, 'lightgcn-ml1m.pt'))
    with torch.no_grad():
        eu, ei = model.readout(adj)
        m = evaluate(eu, ei, test_records, EXCL_TEST, pop, tail_set=tail_test)
    print('\n--- BASELINE LightGCN (test) ---')
    for k in TOP_KS:
        print(f'Recall@{k} {m[k]["recall"]:.4f}  NDCG@{k} {m[k]["ndcg"]:.4f}  '
              f'ARP@{k} {m[k]["arp"]:.4f}')
    print(f'Tail Recall@50 {m["tail_recall@50"]:.4f} '
          f'(over {m["tail_users"]} users with >=1 tail target)')
    return model, adj, pop, m


backbone, sparse_adj, item_pop, baseline_metrics = train_backbone()
TAIL_VAL,  _ = tail_items_for(val_records,  item_pop)
TAIL_TEST, _ = tail_items_for(test_records, item_pop)

[tail] val: 7004 items below pop 0.0056 | test: 6398 items below pop 0.0061
Epoch [1/2000] Loss 0.6609 Recall@50 0.0202 NDCG@50 0.0120 ARP@50 0.2500 11.8s
Epoch [2/2000] Loss 0.4111 Recall@50 0.0203 NDCG@50 0.0117 ARP@50 0.2478 10.5s
Epoch [3/2000] Loss 0.2656 Recall@50 0.0245 NDCG@50 0.0141 ARP@50 0.2233 10.6s
Epoch [4/2000] Loss 0.2103 Recall@50 0.0277 NDCG@50 0.0160 ARP@50 0.2037 10.6s
Epoch [5/2000] Loss 0.1803 Recall@50 0.0305 NDCG@50 0.0176 ARP@50 0.1916 11.0s
Epoch [6/2000] Loss 0.1608 Recall@50 0.0327 NDCG@50 0.0188 ARP@50 0.1829 10.9s
Epoch [7/2000] Loss 0.1470 Recall@50 0.0344 NDCG@50 0.0198 ARP@50 0.1765 11.0s
Epoch [8/2000] Loss 0.1375 Recall@50 0.0364 NDCG@50 0.0208 ARP@50 0.1710 11.0s
Epoch [9/2000] Loss 0.1293 Recall@50 0.0377 NDCG@50 0.0216 ARP@50 0.1666 11.1s
Epoch [10/2000] Loss 0.1229 Recall@50 0.0389 NDCG@50 0.0224 ARP@50 0.1631 11.4s
Epoch [11/2000] Loss 0.1174 Recall@50 0.0400 NDCG@50 0.0231 ARP@50 0.1601 11.2s
Epoch [12/2000] Loss 0.1127 Recall@50 0.0411 NDCG@50 

## PPD implementation notes

PPD is defined by **Algorithm 1** of the paper in three steps:

1. **Popularity estimation** (Eqs. 5-7) — for every *training* interaction
   $(u,i)$, compute a popularity score
   $b_{ui} = p_i - r_{ui}$, where $p_i$ is the *global preference* of item
   $i$ (its average similarity to all users) and $r_{ui}$ is user $u$'s
   *personalized preference* for item $i$ (similarity to $u$'s own history,
   discounted by a popularity-penalty term controlled by $\beta$). Both
   $p_i$ and $r_{ui}$ are min-max normalized to $[0,1]$ before differencing
   (paper, Appendix B.2), and we compute them on the backbone's **stable,
   post-propagation** embeddings, exactly as the paper specifies.
2. **Popularity direction construction** (Eqs. 8-10) — for every user/item
   node, build a *popularity centroid* (neighbors weighted by $b_{ui}$) and a
   *preference centroid* (neighbors weighted by $1-b_{ui}$) from the
   backbone's **layer-0** embeddings, then take
   $\mathbf d_{pop}(v) = \bar{\mathbf e}_{pop}(v) - \phi\,\bar{\mathbf e}_{pref}(v)$.
3. **Debiasing via projection** (Eqs. 11-12) — subtract the projection of
   each layer-0 embedding onto its own popularity direction, then
   **re-propagate** the debiased layer-0 embeddings through the *same*
   normalized adjacency used to train the backbone. No retraining is ever
   performed — PPD is purely post-hoc.

Hyperparameters (paper, Appendix B.2), swept below on the paper's own grid,
selected by NDCG@50 on the balanced validation split (the same selection
rule your own BCPS sweep uses):

* $\beta \in \{0.0, 0.1, 0.2, 0.3\}$ — popularity penalty inside $r_{ui}$.
* $\phi \in \{0.0, 0.25, 0.50, 0.75, 1.0\}$ — preference-centroid weight in
  the popularity direction.
* $\epsilon = 10^{-8}$ — denominator guard in the centroid formulas.

We also reproduce the paper's **RQ5 ablation** ("PPD w/o-PP"), which drops
the personalized-preference term and uses $b_{ui} = p_i$ directly, and the
**dataset-aware PPAC reference table** (transcribed from arXiv:2402.07425,
Table 2) so the "vs PPAC" section is valid whichever `DATASET_NAME` this
notebook is run with.


In [3]:
# ============================================================================
# CELL PPD-1 — PPD: POST-HOC POPULARITY DEBIASING
#   Md Aminul Islam, Elena Zheleva, Ren Wang. "Post-hoc Popularity Bias
#   Correction in GNN-based Collaborative Filtering." WWW'26 (arXiv:2510.12959)
#
# Runs on top of the SAME pre-trained LightGCN backbone (`backbone`,
# `sparse_adj`) and the SAME train/val/test splits, exclusion sets, item
# popularity vector and evaluation protocol built in CELL 1 / CELL 2 above.
# PPD does NOT retrain the backbone: it only edits the layer-0 (l=0)
# embeddings post-hoc and re-propagates them through the already-built
# normalized adjacency `sparse_adj`, using the SAME `propagate()` function
# used to train/evaluate the backbone, so the comparison against your own
# post-hoc BCPS method is apples-to-apples (same backbone, same splits, same
# metrics, same masking).
#
# ---------------------------------------------------------------------------
# Paper recap (Sec. 4, Alg. 1):
#   1) Popularity estimation (Eqs. 5-7), computed on the STABLE, POST-
#      PROPAGATION backbone embeddings (paper: "we compute the popularity
#      score for each interaction using the stable user and item embeddings
#      learned by the pre-trained model"):
#         p_i   = (1/|U|) * sum_u sim(e_u, e_i)                       (5)
#         r_ui  = (1/|N_u|) * sum_{j in N_u} [sim(e_i,e_j) - beta*p_i*p_j]  (6)
#         b_ui  = p_i - r_ui                                          (7)
#      sim(.,.) = dot product (paper, Appendix B.2). p_i and r_ui are
#      min-max normalized to [0,1] before differencing (paper, Appendix B.2).
#   2) Popularity direction construction (Eqs. 8-10), computed on the
#      LAYER-0 (pre-propagation, "l=0") embeddings:
#         e_pop(v)  = weighted centroid of v's neighbours, weight = b_ui
#         e_pref(v) = weighted centroid of v's neighbours, weight = 1-b_ui
#         d_pop(v)  = e_pop(v) - phi * e_pref(v)
#   3) Debiasing via projection (Eqs. 11-12), applied at layer 0, THEN
#      propagated through all L layers of the GCN so unbiased signal flows
#      to every layer:
#         e~_v(0) = e_v(0) - Proj_{d_pop(v)}(e_v(0))
#
# Hyperparameters (paper, Appendix B.2): beta in {0.0,0.1,0.2,0.3} (popularity
# penalty in r_ui), phi in {0.0,0.25,0.50,0.75,1.0} (preference-centroid
# weight in d_pop), epsilon = 1e-8 (denominator guard in Eqs. 8-9).
# ============================================================================
PPD_EPS = 1e-8


def ppd_build_edge_lists(train_recs, device):
    """Flatten the training interaction set E = {(u,i)} into parallel
    (edge_u, edge_i) index tensors, exactly the interactions PPD is defined
    over (Alg. 1, line 5: 'for each interaction (u,i) in E')."""
    us, it = [], []
    for u, items in train_recs.items():
        us.extend([u] * len(items))
        it.extend(items)
    edge_u = torch.tensor(us, dtype=torch.long, device=device)
    edge_i = torch.tensor(it, dtype=torch.long, device=device)
    return edge_u, edge_i


def _minmax(x):
    return (x - x.min()) / (x.max() - x.min() + PPD_EPS)


def ppd_popularity_scores(eu_stable, ei_stable, edge_u, edge_i, num_users, beta):
    """Eqs. (5)-(7): per-interaction popularity score b_ui.

    eu_stable, ei_stable: the backbone's STABLE (post-propagation, final)
    user/item embeddings -- the same embeddings used for scoring
    (y_hat_ui = e_u^T e_i) in the pre-trained model.
    Returns (p_i_norm[num_items], b_edge[num_edges]) with both terms
    min-max normalized to [0,1] (paper, Appendix B.2), and b_edge additionally
    clamped to [0,1]: p_i_norm, r_ui_norm both lie in [0,1] so their raw
    difference lies in [-1,1]; a negative value means the interaction is
    explained by personal preference rather than popularity, which is exactly
    the b_ui -> 0 (=> preference weight 1-b_ui -> 1) limit the paper assigns
    to preference-driven interactions, so we floor at 0 rather than let a
    negative weight corrupt the convex centroids in Eqs. (8)-(9).
    """
    device = eu_stable.device

    # ---- Eq. (5): p_i = mean_u <e_u, e_i> = <e_i, mean_u e_u> ----------------
    u_bar = eu_stable.mean(dim=0)                           # [D]
    p_i = ei_stable @ u_bar                                  # [num_items]

    # ---- per-user aggregates needed to vectorize Eq. (6) over every edge -----
    # e_bar(u) = mean_{j in N_u} e_j ,  p_bar(u) = mean_{j in N_u} p_j
    D = ei_stable.shape[1]
    deg_u = torch.zeros(num_users, device=device).index_add_(
        0, edge_u, torch.ones_like(edge_u, dtype=torch.float32)).clamp(min=1.0)
    e_bar_u = torch.zeros(num_users, D, device=device).index_add_(
        0, edge_u, ei_stable[edge_i])
    e_bar_u = e_bar_u / deg_u.unsqueeze(1)
    p_bar_u = torch.zeros(num_users, device=device).index_add_(
        0, edge_u, p_i[edge_i]) / deg_u

    # ---- Eq. (6): r_ui = <e_i, e_bar(u)> - beta * p_i * p_bar(u) -------------
    r_ui = ((ei_stable[edge_i] * e_bar_u[edge_u]).sum(dim=1)
             - beta * p_i[edge_i] * p_bar_u[edge_u])

    # ---- normalize, then Eq. (7): b_ui = p_i - r_ui ---------------------------
    p_i_n = _minmax(p_i)
    r_ui_n = _minmax(r_ui)
    b_edge = (p_i_n[edge_i] - r_ui_n).clamp(min=0.0, max=1.0)

    return p_i_n, b_edge


def ppd_popularity_scores_wo_pp(eu_stable, ei_stable, edge_i):
    """RQ5 ablation, 'PPD w/o-PP': popularity score uses ONLY global
    preference p_i (min-max normalized), ignoring personalized preference
    entirely (paper Sec 5.2 / Table 5)."""
    u_bar = eu_stable.mean(dim=0)
    p_i = ei_stable @ u_bar
    p_i_n = _minmax(p_i)
    return p_i_n, p_i_n[edge_i].clamp(min=0.0, max=1.0)


def ppd_direction_vectors(eu0, ei0, edge_u, edge_i, b_edge, num_users, num_items, phi):
    """Eqs. (8)-(10): popularity/preference centroids (on LAYER-0 embeddings)
    and the resulting popularity direction vector, for every user and item
    node that has at least one training interaction. Nodes with none get a
    zero direction vector and are left untouched by the projection step."""
    device = eu0.device
    D = eu0.shape[1]
    w_pop = b_edge
    w_pref = 1.0 - b_edge

    # ---- user centroids, Eq. (8): aggregate over items j in N(u) -------------
    num_u = torch.zeros(num_users, device=device).index_add_(0, edge_u, w_pop)
    den_u = torch.zeros(num_users, device=device).index_add_(0, edge_u, w_pref)
    pop_u = torch.zeros(num_users, D, device=device).index_add_(
        0, edge_u, ei0[edge_i] * w_pop.unsqueeze(1))
    pref_u = torch.zeros(num_users, D, device=device).index_add_(
        0, edge_u, ei0[edge_i] * w_pref.unsqueeze(1))
    e_pop_u = pop_u / (num_u.unsqueeze(1) + PPD_EPS)
    e_pref_u = pref_u / (den_u.unsqueeze(1) + PPD_EPS)

    # ---- item centroids, Eq. (9): aggregate over users u in N(i) -------------
    num_i = torch.zeros(num_items, device=device).index_add_(0, edge_i, w_pop)
    den_i = torch.zeros(num_items, device=device).index_add_(0, edge_i, w_pref)
    pop_i = torch.zeros(num_items, D, device=device).index_add_(
        0, edge_i, eu0[edge_u] * w_pop.unsqueeze(1))
    pref_i = torch.zeros(num_items, D, device=device).index_add_(
        0, edge_i, eu0[edge_u] * w_pref.unsqueeze(1))
    e_pop_i = pop_i / (num_i.unsqueeze(1) + PPD_EPS)
    e_pref_i = pref_i / (den_i.unsqueeze(1) + PPD_EPS)

    # ---- Eq. (10): popularity direction vector --------------------------------
    d_u = e_pop_u - phi * e_pref_u
    d_i = e_pop_i - phi * e_pref_i
    return d_u, d_i


def ppd_project_remove(e0, d):
    """Eqs. (11)-(12): remove the popularity-aligned component of e0 via
    vector projection onto d, per node. Nodes with a (near-)zero-norm
    direction vector (e.g. no training interactions) are left unchanged,
    since the projection is undefined there."""
    dot = (e0 * d).sum(dim=1, keepdim=True)
    norm_sq = (d * d).sum(dim=1, keepdim=True)
    has_dir = norm_sq > PPD_EPS
    coeff = torch.where(has_dir, dot / norm_sq.clamp(min=PPD_EPS), torch.zeros_like(dot))
    return e0 - coeff * d


@torch.no_grad()
def run_ppd(beta, phi, with_personalized_preference=True,
            backbone=backbone, adj=sparse_adj, n_layers=N_LAYERS):
    """Full PPD pipeline (Algorithm 1): estimate b_ui on the pre-trained
    backbone's STABLE embeddings -> build per-node popularity direction on
    the LAYER-0 embeddings -> project-and-remove at layer 0 -> re-propagate
    through the SAME normalized adjacency used to train/evaluate the
    backbone -> evaluate with the notebook's own evaluate() (same exclusion
    sets, same tail split, same TOP_KS)."""
    backbone.eval()
    eu0 = backbone.user_embedding.weight.detach()
    ei0 = backbone.item_embedding.weight.detach()

    # stable (post-propagation) embeddings used only for popularity ESTIMATION
    eu_stable, ei_stable = propagate(adj, eu0, ei0, n_layers=n_layers)

    edge_u, edge_i = ppd_build_edge_lists(train_records, eu0.device)
    if with_personalized_preference:
        _, b_edge = ppd_popularity_scores(eu_stable, ei_stable, edge_u, edge_i,
                                          NUM_USERS, beta)
    else:
        _, b_edge = ppd_popularity_scores_wo_pp(eu_stable, ei_stable, edge_i)

    d_u, d_i = ppd_direction_vectors(eu0, ei0, edge_u, edge_i, b_edge,
                                     NUM_USERS, NUM_ITEMS, phi)

    eu0_tilde = ppd_project_remove(eu0, d_u)
    ei0_tilde = ppd_project_remove(ei0, d_i)

    # propagate the DEBIASED layer-0 embeddings through all L layers, exactly
    # as the paper prescribes ("...then propagate information...through all
    # subsequent layers of the GCN")
    eu_ppd, ei_ppd = propagate(adj, eu0_tilde, ei0_tilde, n_layers=n_layers)

    m_test = evaluate(eu_ppd, ei_ppd, test_records, EXCL_TEST, item_pop, tail_set=TAIL_TEST)
    m_val = evaluate(eu_ppd, ei_ppd, val_records, EXCL_VAL, item_pop, tail_set=TAIL_VAL)
    m_test['val_ndcg'] = m_val[50]['ndcg']
    return (eu_ppd, ei_ppd), m_test


In [4]:
# ============================================================================
# CELL PPD-2 — HYPERPARAMETER SWEEP, RQ5 ABLATION, PPAC COMPARISON,
#              FINAL COMPARISON TABLE  (dataset-aware: gowalla / yelp2018)
#
# Selection criterion: NDCG@50 on the (balanced) validation split -- the SAME
# selection criterion your own BCPS sweep (CELL 6) uses for `runs[k]['val_ndcg']`,
# so "PPD (best)" and "BCPS (best)" are chosen the same way before comparison.
#
# Grids follow the PPD paper's own hyperparameter search (Appendix B.2):
#   beta in {0.0, 0.1, 0.2, 0.3}      (popularity penalty in r_ui, Eq. 6)
#   phi  in {0.0, 0.25, 0.5, 0.75, 1.0}   (preference-centroid weight, Eq. 10)
# ============================================================================
PPD_BETA_GRID = [0.0, 0.1, 0.2, 0.3]
PPD_PHI_GRID  = [0.0, 0.25, 0.5, 0.75, 1.0]

# ---- PPAC (WWW'24) reference numbers, all three datasets (arXiv:2402.07425,
# Table 2, 'LightGCN' block) -- same table your own CELL 6/7 uses, kept here
# too so this PPD notebook is runnable/reportable standalone. ----------------
PPAC_REFERENCE = {
    'ml-1m':    {'base': {'recall': 0.3757, 'ndcg': 0.2295},
                 'method': {'recall': 0.4056, 'ndcg': 0.2481}},
    'gowalla':  {'base': {'recall': 0.1480, 'ndcg': 0.0544},
                 'method': {'recall': 0.1885, 'ndcg': 0.0780}},
    'yelp2018': {'base': {'recall': 0.0852, 'ndcg': 0.0326},
                 'method': {'recall': 0.1031, 'ndcg': 0.0414}},
}


def ppd_row(name, m, ref=None):
    s = (f'{name:<32} R@50 {m[50]["recall"]:.4f}  N@50 {m[50]["ndcg"]:.4f}  '
         f'ARP@50 {m[50]["arp"]:.4f}  tail {m["tail_recall@50"]:.4f}')
    if ref is not None:
        s += (f'   vs base {100*(m[50]["ndcg"]/ref[50]["ndcg"]-1):+5.1f}% N '
              f'{100*(m[50]["arp"]/ref[50]["arp"]-1):+5.1f}% ARP')
    return s


print(f'=== PPD sweep on {DATASET_NAME!r} (selection on balanced-val NDCG@50) ===')
ppd_runs = {}
for beta in PPD_BETA_GRID:
    for phi in PPD_PHI_GRID:
        _, m = run_ppd(beta=beta, phi=phi)
        ppd_runs[(beta, phi)] = m
        print(ppd_row(f'  beta={beta} phi={phi}', m, baseline_metrics))

PPD_BEST = max(ppd_runs, key=lambda k: ppd_runs[k]['val_ndcg'])
PB_BETA, PB_PHI = PPD_BEST
ppd_best_metrics = ppd_runs[PPD_BEST]
print(f'\n>>> PPD best by VAL NDCG@50: beta={PB_BETA}  phi={PB_PHI}')

# ---- RQ5 ablation: PPD vs PPD w/o personalized preference (paper Table 5) ---
_, ppd_wopp_metrics = run_ppd(beta=PB_BETA, phi=PB_PHI, with_personalized_preference=False)

print('\n=== PPD vs backbone vs RQ5 ablation ===')
print(ppd_row('0. LightGCN baseline', baseline_metrics))
print(ppd_row('1. PPD (best beta, phi)', ppd_best_metrics, baseline_metrics))
print(ppd_row('   PPD w/o-PP (RQ5)', ppd_wopp_metrics, baseline_metrics))

# ---- vs PPAC (dataset-aware) -------------------------------------------------
if DATASET_NAME not in PPAC_REFERENCE:
    print(f'\n(no published PPAC reference for DATASET_NAME={DATASET_NAME!r}; '
          f'skipping PPAC comparison block)')
else:
    ref = PPAC_REFERENCE[DATASET_NAME]
    print(f"\n=== vs PPAC (WWW'24) Table 2, {DATASET_NAME} / LightGCN ===")
    print(f'{"":<30}{"R@50":>9}{"N@50":>9}')
    print(f'{"PPAC: LightGCN base":<30}{ref["base"]["recall"]:>9.4f}{ref["base"]["ndcg"]:>9.4f}')
    print(f'{"PPAC: their method":<30}{ref["method"]["recall"]:>9.4f}{ref["method"]["ndcg"]:>9.4f}')
    print(f'{"ours: LightGCN base":<30}{baseline_metrics[50]["recall"]:>9.4f}'
          f'{baseline_metrics[50]["ndcg"]:>9.4f}')
    print(f'{"ours: PPD":<30}{ppd_best_metrics[50]["recall"]:>9.4f}{ppd_best_metrics[50]["ndcg"]:>9.4f}')
    gp_ppac = 100 * (ref['method']['ndcg'] / ref['base']['ndcg'] - 1)
    gp_ours = 100 * (ppd_best_metrics[50]['ndcg'] / baseline_metrics[50]['ndcg'] - 1)
    print('\nRELATIVE GAIN OVER OWN BASELINE (the only fair comparison, since '
          'this split protocol differs from PPAC\'s own):')
    print(f'  PPAC : {100*(ref["method"]["recall"]/ref["base"]["recall"]-1):+5.1f}% R  {gp_ppac:+5.1f}% N')
    print(f'  PPD  : {100*(ppd_best_metrics[50]["recall"]/baseline_metrics[50]["recall"]-1):+5.1f}% R  {gp_ours:+5.1f}% N')

# ---- final head-to-head table: baseline vs PPD vs your own post-hoc method --
# (BCPS / `m_best` is populated once your own pipeline's sweep cell has run
# in the SAME kernel; this block degrades gracefully if it hasn't.)
print(f'\n=== FINAL COMPARISON on {DATASET_NAME!r} (test split) ===')
header = f'{"":<28}{"R@20":>8}{"N@20":>8}{"R@50":>8}{"N@50":>8}{"ARP@50":>9}{"tail@50":>9}'
print(header)


def _fmt(name, m):
    print(f'{name:<28}{m[20]["recall"]:>8.4f}{m[20]["ndcg"]:>8.4f}'
          f'{m[50]["recall"]:>8.4f}{m[50]["ndcg"]:>8.4f}'
          f'{m[50]["arp"]:>9.4f}{m["tail_recall@50"]:>9.4f}')


_fmt('LightGCN (no debiasing)', baseline_metrics)
_fmt(f'PPD (beta={PB_BETA}, phi={PB_PHI})', ppd_best_metrics)
try:
    _fmt('BCPS / your method (best)', m_best)   # defined in your own CELL 6, if run
except NameError:
    print('(BCPS `m_best` not found in this kernel -- run your own pipeline '
          "cells first to populate it, or compare ppd_best_metrics against "
          "your saved results offline.)")

print('\nRelative gain over the SAME shared LightGCN backbone:')
gp = 100 * (ppd_best_metrics[50]['ndcg'] / baseline_metrics[50]['ndcg'] - 1)
print(f'  PPD  : {100*(ppd_best_metrics[50]["recall"]/baseline_metrics[50]["recall"]-1):+5.1f}% R@50  {gp:+5.1f}% N@50')
try:
    gb = 100 * (m_best[50]['ndcg'] / baseline_metrics[50]['ndcg'] - 1)
    print(f'  BCPS : {100*(m_best[50]["recall"]/baseline_metrics[50]["recall"]-1):+5.1f}% R@50  {gb:+5.1f}% N@50')
except NameError:
    pass

# ---- persist raw numbers for the paper's tables/plots -----------------------
import json as _json
ppd_export = {
    'dataset': DATASET_NAME,
    'grid': {'beta': PPD_BETA_GRID, 'phi': PPD_PHI_GRID},
    'best': {'beta': PB_BETA, 'phi': PB_PHI},
    'sweep': {f'{b}|{p}': {k: v for k, v in ppd_runs[(b, p)].items()
                            if k in TOP_KS + ['tail_recall@50', 'val_ndcg']}
              for (b, p) in ppd_runs},
    'best_metrics': {k: ppd_best_metrics[k] for k in TOP_KS + ['tail_recall@50', 'val_ndcg']},
    'wo_pp_metrics': {k: ppd_wopp_metrics[k] for k in TOP_KS + ['tail_recall@50', 'val_ndcg']},
    'baseline_metrics': {k: baseline_metrics[k] for k in TOP_KS + ['tail_recall@50']},
    'ppac_reference': PPAC_REFERENCE.get(DATASET_NAME),
}
_out_path = os.path.join(OUT_ROOT, f'ppd_results_{DATASET_NAME}.json')
with open(_out_path, 'w') as fh:
    _json.dump(ppd_export, fh, indent=2, default=str)
print(f'\n[saved] {_out_path}')


=== PPD sweep on 'gowalla' (selection on balanced-val NDCG@50) ===
  beta=0.0 phi=0.0               R@50 0.0355  N@50 0.0249  ARP@50 0.1139  tail 0.0120   vs base -75.8% N +38.5% ARP
  beta=0.0 phi=0.25              R@50 0.0335  N@50 0.0218  ARP@50 0.1330  tail 0.0102   vs base -78.8% N +61.7% ARP
  beta=0.0 phi=0.5               R@50 0.0614  N@50 0.0421  ARP@50 0.0949  tail 0.0238   vs base -59.1% N +15.4% ARP
  beta=0.0 phi=0.75              R@50 0.1131  N@50 0.0822  ARP@50 0.0461  tail 0.0567   vs base -20.2% N -43.9% ARP
  beta=0.0 phi=1.0               R@50 0.1019  N@50 0.0702  ARP@50 0.0988  tail 0.0399   vs base -31.9% N +20.1% ARP
  beta=0.1 phi=0.0               R@50 0.0356  N@50 0.0250  ARP@50 0.1139  tail 0.0120   vs base -75.8% N +38.5% ARP
  beta=0.1 phi=0.25              R@50 0.0334  N@50 0.0218  ARP@50 0.1329  tail 0.0102   vs base -78.8% N +61.6% ARP
  beta=0.1 phi=0.5               R@50 0.0615  N@50 0.0422  ARP@50 0.0948  tail 0.0241   vs base -59.1% N +15.2% ARP
  bet